In [4]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import math
from datetime import datetime

In [ ]:
# 1. Configuration & Setup
VIDEO_PATH = 'petrol-pump_compress.mp4'  # Replace with your video file path
OUTPUT_PATH = 'output_video.mp4'
LOG_FILE = 'activity_log.csv'

In [5]:
model = YOLO('yolov8n.pt')

In [6]:
ROI_PUMP_LEFT = np.array([[380, 150], [480, 150], [480, 350], [380, 350]], np.int32)
ROI_PUMP_RIGHT = np.array([[550, 150], [680, 150], [680, 350], [550, 350]], np.int32)
ROIS = [ROI_PUMP_LEFT, ROI_PUMP_RIGHT]

In [ ]:
track_history = {} 
MAX_HISTORY = 30  
STATIONARY_THRESHOLD = 20 
activity_logs = []

In [10]:
# 2. Helper Functions
def calculate_movement(history):
    """Calculates the distance moved between the oldest and newest point in history."""
    if len(history) < 2:
        return 0.0
    start = history[0]
    end = history[-1]
    return math.dist(start, end)

def check_in_roi(point, rois):
    """Checks if a point (x,y) is inside any of the defined ROIs."""
    for roi in rois:
        # pointPolygonTest returns +1 if inside, 0 if on edge, -1 if outside
        if cv2.pointPolygonTest(roi, point, False) >= 0:
            return True
    return False

In [16]:
cap = cv2.VideoCapture('petrol-pump.mp4')
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    frame_count += 1
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    results = model.track(frame, persist=True, tracker="botsort.yaml", verbose=False)

    for roi in ROIS:
        cv2.polylines(frame, [roi], isClosed=True, color=(255, 0, 0), thickness=2)
        cv2.putText(frame, "Workspace (Pump)", (roi[0][0], roi[0][1] - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()

        for box, track_id, cls in zip(boxes, track_ids, classes):
            # Process only class 0 (Person)
            if int(cls) != 0:
                continue
                
            x1, y1, x2, y2 = map(int, box)
            
            # Use the bottom-center of the bounding box (where the feet are) for ROI checks
            bottom_center = (int((x1 + x2) / 2), y2)
            
            # Update track history
            track_id = int(track_id)
            if track_id not in track_history:
                track_history[track_id] = []
            track_history[track_id].append(bottom_center)
            
            if len(track_history[track_id]) > MAX_HISTORY:
                track_history[track_id].pop(0)

           
            is_in_workspace = check_in_roi(bottom_center, ROIS)
            movement_distance = calculate_movement(track_history[track_id])
            is_stationary = movement_distance < STATIONARY_THRESHOLD

           
            if is_in_workspace and is_stationary:
                state = "Working"
                color = (0, 255, 0)  # Green
            elif not is_in_workspace and not is_stationary:
                state = "Moving"
                color = (0, 165, 255) # Orange
            else:
                state = "Idle"
                color = (0, 0, 255)  # Red

           
            if frame_count % fps == 0:
                activity_logs.append({
                    "Timestamp": current_time,
                    "Person_ID": track_id,
                    "Activity_State": state,
                    "In_ROI": is_in_workspace
                })

           
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label = f"ID: {track_id} | {state}"
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # Draw movement trail
            points = np.hstack(track_history[track_id]).astype(np.int32).reshape((-1, 1, 2))
            cv2.polylines(frame, [points], isClosed=False, color=(255, 255, 0), thickness=1)

    out.write(frame)

In [15]:
#6. Clean Up & Save Data
# ==========================================
cap.release()
out.release()
cv2.destroyAllWindows()

# Export logs to CSV
df = pd.DataFrame(activity_logs)
df.to_csv(LOG_FILE, index=False)
print(f"Processing complete. Video saved to {OUTPUT_PATH} and logs saved to {LOG_FILE}.")

Processing complete. Video saved to output_video.mp4 and logs saved to activity_log.csv.
